# Data Cleaning | Retail Customer Churn & Revenue Analysis

This notebook documents the reproducible cleaning decisions applied to the public IBM Telco Customer Churn dataset. The raw file has 7,043 customer records and 21 source columns. The project treats the dataset as a customer-level snapshot rather than a dated transaction fact table.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
raw_path = ROOT / 'data/raw/Telco-Customer-Churn.csv'
clean_path = ROOT / 'data/cleaned/telco_customer_churn_cleaned.csv'
raw = pd.read_csv(raw_path)
cleaned = pd.read_csv(clean_path)
raw.shape, cleaned.shape

## Data-quality checks

The source contains 11 blank `TotalCharges` values. These rows correspond to customers with zero months of tenure, so the cleaning rule imputes `TotalCharges = 0`. Exact duplicates and IQR outliers are checked explicitly rather than silently removed.

In [ ]:
quality = {
    'raw_rows': len(raw),
    'exact_duplicate_rows': int(raw.duplicated().sum()),
    'blank_total_charges': int(raw['TotalCharges'].astype(str).str.strip().eq('').sum()),
    'cleaned_rows': len(cleaned),
    'cleaned_percent_of_raw': round(len(cleaned) / len(raw) * 100, 2),
}
quality

In [ ]:
numeric_review = cleaned[['tenure', 'monthly_charges', 'total_charges']].describe().T
numeric_review

## Re-run the production pipeline

The full, source-controlled implementation is in `src/build_analysis.py`. Running the following cell refreshes the cleaned CSV, cleaning log, metrics tables, and visuals.

In [ ]:
# In a terminal from the repository root: python src/build_analysis.py
with open(ROOT / 'data/cleaned/cleaning_log.json') as f:
    cleaning_log = json.load(f)
cleaning_log